# Databricks vs Snowflake Funding History

This demo notebook creates a small dataset with the funding history of **Databricks** and **Snowflake**.  
We will apply a simple transformation to calculate the **cumulative funding raised over time** for each company.

In [2]:

from pyspark.sql import Row

# Publicly available funding data (simplified, USD millions)
funding_data = [
    # Databricks funding rounds
    Row(company="Databricks", round="Seed",      year=2013, amount_million=13.9),
    Row(company="Databricks", round="Series A",  year=2014, amount_million=33.0),
    Row(company="Databricks", round="Series B",  year=2015, amount_million=60.0),
    Row(company="Databricks", round="Series C",  year=2016, amount_million=140.0),
    Row(company="Databricks", round="Series D",  year=2017, amount_million=140.0),
    Row(company="Databricks", round="Series E",  year=2019, amount_million=250.0),
    Row(company="Databricks", round="Series F",  year=2021, amount_million=1000.0),
    Row(company="Databricks", round="Series G",  year=2022, amount_million=1600.0),

    # Snowflake funding rounds
    Row(company="Snowflake", round="Series A",   year=2012, amount_million=5.0),
    Row(company="Snowflake", round="Series B",   year=2014, amount_million=26.0),
    Row(company="Snowflake", round="Series C",   year=2015, amount_million=80.0),
    Row(company="Snowflake", round="Series D",   year=2017, amount_million=100.0),
    Row(company="Snowflake", round="Series E",   year=2018, amount_million=263.0),
    Row(company="Snowflake", round="Series F",   year=2020, amount_million=479.0),
]

df = spark.createDataFrame(funding_data)

display(df)

,company,round,year,amount_million
0,Databricks,Seed,2013,13.9
1,Databricks,Series A,2014,33.0
2,Databricks,Series B,2015,60.0
3,Databricks,Series C,2016,140.0
4,Databricks,Series D,2017,140.0
5,Databricks,Series E,2019,250.0
6,Databricks,Series F,2021,1000.0
7,Databricks,Series G,2022,1600.0
8,Snowflake,Series A,2012,5.0
9,Snowflake,Series B,2014,26.0


## Step 2: Calculate Cumulative Funding

Now we calculate the **cumulative funding raised** over time for each company.  
This will let us see how the total funding grew year by year.


In [3]:
from pyspark.sql import Window
from pyspark.sql import functions as F

df_transformed = (
    df.orderBy("company", "year")
      .withColumn(
          "cumulative_million",
          F.sum("amount_million").over(
              Window.partitionBy("company").orderBy("year").rowsBetween(Window.unboundedPreceding, Window.currentRow)
          )
      )
)

display(df_transformed)

,company,round,year,amount_million,cumulative_million
0,Databricks,Seed,2013,13.9,13.9
1,Databricks,Series A,2014,33.0,46.9
2,Databricks,Series B,2015,60.0,106.9
3,Databricks,Series C,2016,140.0,246.9
4,Databricks,Series D,2017,140.0,386.9
5,Databricks,Series E,2019,250.0,636.9
6,Databricks,Series F,2021,1000.0,1636.9
7,Databricks,Series G,2022,1600.0,3236.9
8,Snowflake,Series A,2012,5.0,5.0
9,Snowflake,Series B,2014,26.0,31.0
